In [ ]:
import xarray as xr

# Load your Feb–March dataset
file_path = r"yourfilepath.nc"

ds = xr.open_dataset(file_path)
print(ds)

In [ ]:
ds["t2m_c"] = ds["t2m"] - 273.15
temp_series = ds["t2m_c"]
df = temp_series.to_dataframe().reset_index()


In [ ]:
import pandas as pd

df = temp_series.to_dataframe().reset_index()
df.head()


In [ ]:
for i in range(1, 25):
    df[f"temp_t-{i}"] = df["t2m_c"].shift(i)

In [ ]:
df["target"] = df["t2m_c"].shift(-24)
df = df.dropna()

In [ ]:
X = df[[f"temp_t-{i}" for i in range(1, 25)]]
y = df["target"]

In [ ]:
import joblib

lr = joblib.load("linear_regression.pkl")
rf = joblib.load("random_forest.pkl")
gbr = joblib.load("gradient_boosting.pkl")
xgb = joblib.load("xgboost.pkl")
svr = joblib.load("svr.pkl")

models = {
    "Linear Regression": lr,
    "Random Forest": rf,
    "Gradient Boosting": gbr,
    "XGBoost": xgb,
    "SVR": svr
}

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

results = {}

for name, model in models.items():
    pred = model.predict(X)
    
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    
    results[name] = {"MAE": mae, "RMSE": rmse}
    
    print(f"{name} -> MAE: {mae:.3f}, RMSE: {rmse:.3f}")

In [ ]:
best_model_name = min(results, key=lambda x: results[x]["MAE"])
best_model = models[best_model_name]

print("\nBest Model:", best_model_name)
print("MAE:", results[best_model_name]["MAE"])

In [ ]:
import joblib

joblib.dump(best_model, "best_model_feb_march.pkl")

print("Saved best model:", best_model_name)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_all(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

In [ ]:
results = {}

for name, model in models.items():
    pred = model.predict(X)
    results[name] = evaluate_all(y, pred)

import pandas as pd
df_results = pd.DataFrame(results).T

print(df_results)

In [ ]:
import joblib

joblib.dump(lr, "final_best_model.pkl")

print("Final model saved successfully!")